In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import copy
import hydra
from hydra import initialize, compose
from omegaconf import DictConfig

from omniisaacgymenvs.utils.hydra_cfg.hydra_utils import *
from omniisaacgymenvs.utils.hydra_cfg.reformat import omegaconf_to_dict, print_dict

from omniisaacgymenvs.utils.task_util import initialize_task
from omniisaacgymenvs.envs.vec_env_rlgames import VecEnvRLGames

In [2]:
from os.path import join, dirname
import os
import sys
import time

import torch
from torch import nn, einsum
import torch.nn.functional as F
from torch.distributions import Normal

from einops import rearrange, repeat

sys.path.append(join(dirname(dirname(os.getcwd()))))
sys.path.append(os.getcwd())

from memoryPerceiver.memory_perceiver.main.configuration import cf

from memoryPerceiver.memory_perceiver.core.model_net import PerceiverIO
from memoryPerceiver.memory_perceiver.processing.vectorize import input_to_tensor, fourier_encode

%load_ext autoreload
%autoreload 2
%env ISAAC_JUPYTER_KERNEL=1

env: ISAAC_JUPYTER_KERNEL=1


/isaac-sim/kit/python/lib/python3.7/site-packages/IPython/extensions/autoreload.py:121: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  from imp import reload


In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
random.seed(0)
np.random.seed(0)
torch.use_deterministic_algorithms(False)
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


In [4]:
nb_frames = 32
frames_future = 8
len_video_frame = 1824  # 456*256/(8*8)
len_audio_frame = 120   # 1920/16
query_dim = 832
video_latent_dim = 256
audio_latent_dim = 560
batch_size = 1

len_vid = nb_frames*len_video_frame
len_audio = nb_frames*len_audio_frame
len_audio_raw = len_audio*cf.sample_per_frame
video_future = frames_future*len_video_frame
audio_future = frames_future*len_audio_frame

special_compression = [i for i in range(video_future)] + [i for i in range(len_vid, len_vid+audio_future)]
for k in range(len(cf.body_parts)):
    special_compression += [ len_vid + len_audio + k*nb_frames + i for i in range(frames_future) ]

In [5]:
def create_model( device=device):

    # Projection to return modalities, enhancement

    modality_parameters = {}
    modality_parameters['audio'] = audio_latent_dim
    modality_parameters['video'] = video_latent_dim
    for part, params in cf.body_parts.items():
        modality_parameters[part] = query_dim - 2*128 - params
    for part, params in cf.body_outputs.items():
        modality_parameters[part] = query_dim - 2*128 - params

    projection_parameters = dict(cf.body_outputs)
    projection_parameters.update({'audio': 16})

    # Instantiate model
        
    model = PerceiverIO(
            2,
            query_dim,
            special_compression = special_compression,
            device=device,
            projection_parameters = projection_parameters,
            modality_parameters = modality_parameters,
            max_batch_size = batch_size
        )
    model = model.to(device)

    # Queries

    queries = []
    audio_query = repeat( torch.cat([model.modality['audio'], torch.zeros(16, device=device)]), '... -> n ...', n=audio_future)
    audio_pos = fourier_encode(torch.linspace(-1, 1, audio_future), audio_future/2, num_bands=128).to(device)
    queries.append(torch.cat([audio_query, audio_pos], dim=-1).to(device))
    for part, params in cf.body_outputs.items():
        part_query = repeat( torch.cat([model.modality[part], torch.zeros(params, device=device)]), '... -> n ...', n=frames_future)
        part_pos = fourier_encode(torch.linspace(-1, 1, frames_future), frames_future/2, num_bands=128).to(device)
        queries.append(torch.cat([part_query, part_pos], dim=-1).to(device))
    model.queries=torch.cat(queries)


    return model

In [6]:
# model = create_model()
# model_student = create_model()
model = nn.Sequential(
                        nn.Linear(4, 64),
                        nn.ReLU(),
                        nn.Linear(64, 64),
                        nn.ReLU(),
                        nn.Linear(64, 2)
                    ).to(device)
model_student = nn.Sequential(
                        nn.Linear(4, 64),
                        nn.ReLU(),
                        nn.Linear(64, 64),
                        nn.ReLU(),
                        nn.Linear(64, 2)
                    ).to(device)
model_critic = nn.Sequential(
                        nn.Linear(4, 64),
                        nn.ReLU(),
                        nn.Linear(64, 64),
                        nn.ReLU(),
                        nn.Linear(64, 1)
                    ).to(device)
model_critic_student = nn.Sequential(
                        nn.Linear(4, 64),
                        nn.ReLU(),
                        nn.Linear(64, 64),
                        nn.ReLU(),
                        nn.Linear(64, 1)
                    ).to(device)

# inputs={}
# inputs['video']=torch.randn(batch_size, 3, nb_frames, 256, 456)
# inputs['audio']=torch.randn(batch_size, nb_frames*1920)
# inputs['observations']=torch.randn(batch_size, nb_frames, 4)
# inputs['returns']=torch.randn(batch_size, nb_frames, 1)
# batch = input_to_tensor(inputs, model.modality, cf.body_parts).to(device)
# model(batch, queries = model.queries, update_mem=False)

In [7]:
#@hydra.main(config_name="config", config_path="../cfg")
#@hydra.main(config_name="config", config_path="../../OmniIsaacGymEnvs/omniisaacgymenvs/cfg")
def parse_hydra_configs(cfg: DictConfig):

    def buffer_padding(buffer, dim, default_value=None):

        # zero padding then transform: t b (d) -> b t (d)
        pad0 = np.max([0, nb_frames-len(buffer)])
        if default_value is None:
            padding = torch.zeros([pad0, numEnvs]+dim, device=device)
        else:
            padding = repeat(default_value, ' ... -> t0 b ...', t0=pad0, b=numEnvs).to(device)
        res = torch.swapaxes( 
                torch.cat([ padding, torch.stack(buffer) ])
                , 0, 1)
        return res

    def update_buffer(buff, key, element):
        buff[key].append(element)
        if(len(buff[key])>nb_frames):
            buff[key].pop(0)

    def calculate_returns(rewards_mem, death_index, gamma=0.99):
        returns_estimates = torch.zeros_like(rewards_mem)
        for env_idx in range(numEnvs):
            sum = 0.0
            for t in reversed(range(sim_duration)):
                # If the agent is reset, set to 0
                if t in death_index[env_idx]:
                    sum = 0

                # Reverse calcualtion of returns
                sum = gamma*sum + rewards_mem[t, env_idx]
                returns_estimates[t, env_idx] = sum
            
        return returns_estimates



    def calculate_gae(rewards_mem, values_mem, death_index, gamma=0.99, gae_lambda=0.95):
        sim_duration, numEnvs = rewards_mem.size()
        gae_estimates = torch.zeros_like(rewards_mem)

        for env_idx in range(numEnvs):
            gae = 0.0 
            for t in reversed(range(sim_duration)):
                # If the agent is reset, set GAE to 0
                if t in death_index[env_idx]:
                    gae = 0.0
                    
                # Calculate the TD error and gae
                if t == sim_duration-1:
                    delta_t = values_mem[t, env_idx]  
                else:
                    delta_t= rewards_mem[t, env_idx] + gamma * values_mem[t+1, env_idx] - values_mem[t, env_idx]
                gae_estimates[t, env_idx] = delta_t + gamma * gae_lambda * gae
            
        return gae_estimates
    
    def two_levels_plot(data, ax, fig, horizon, name):
        ax.clear()
        recent_data=data[-horizon:]
        ax.plot(recent_data, alpha=0.5)
        ax.plot((np.convolve(data, np.ones(horizon), 'valid') / horizon)[::max(1,int(len(data)/horizon))])
        # ax.set_ylim(bottom=np.min(recent_data), top=np.max(recent_data))
        fig.savefig('/isaac-sim/memoryPerceiver/memory_perceiver/main/' + name + '.png')



    cfg_dict = omegaconf_to_dict(cfg)

    headless = cfg.headless
    render = not headless
    render = True

    env = VecEnvRLGames(headless=headless, sim_device=cfg.device_id)
    # sets seed. if seed is -1 will pick a random one
    from omni.isaac.core.utils.torch.maths import set_seed
    cfg.seed = set_seed(cfg.seed, torch_deterministic=cfg.torch_deterministic)
    cfg_dict['seed'] = cfg.seed
    task = initialize_task(cfg_dict, env)

    history = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[] ,'returns':[], 'returns_discount':[],'values': []}
    buffer = {'video':[], 'audio':[], 'obs':[], 'reward':[], 'action':[], 'returns':[], 'returns_discount':[], 'returns_goal':[], 'advantages':[]}
    replay_training=True  # Will switch to false immediately
    memories = {i:None for i in range(numEnvs)}
    rewards_mem = torch.tensor([], device=device)
    death_index = {i:[] for i in range(numEnvs)}
    cumulative_reward = torch.zeros(numEnvs, 1, device=device)
    last_batch2=None
    first_run=True
    all_losses=[]
    all_crit=[]
    all_returns=[]
    fig_loss, ax_loss = plt.subplots(figsize=(8, 6))
    fig_crit, ax_crit = plt.subplots(figsize=(8, 6))
    fig_rewards, ax_rewards = plt.subplots(figsize=(8, 6))
    sim_count = 0
    sim_step = 0
    while env._simulation_app.is_running():
        if env._world.is_playing():
            # if env._world.current_time_step_index == 0 or \
            # env._world.current_time_step_index % sim_duration == 0:
            if env.sim_frame_count % sim_duration == 0:

                ### Initialization

                if sim_count % 2 == 1 and len(all_losses)>0:
                    two_levels_plot(all_losses, ax_loss, fig_loss, horizon=160, name='loss')
                    two_levels_plot(all_crit, ax_crit, fig_crit, horizon=160, name='loss_critic')
                    ax_rewards.plot(all_returns)
                    fig_rewards.savefig('/isaac-sim/memoryPerceiver/memory_perceiver/main/returns.png')
                env._world.reset(soft=False)
                replay_training = not replay_training
                # print('$$$$$$$$$$$ CUMUL REWARD $$$$$$$$$$$')
                # print(cumulative_reward)
                if replay_training:
                    print('REPLAY')
                    model.load_state_dict(torch.load(model_path))
                    # model.reset_memories()
                    # model_student.reset_memories()
                    model.eval()
                    model_student.train() 
                    optimizer = torch.optim.Adam(model_student.parameters(), lr=learning_rate)
                    optimizer_critic = torch.optim.Adam(model_critic.parameters(), lr=learning_rate_critic)
                    optimizer.zero_grad()
                    optimizer_critic.zero_grad()
                    values_mem = torch.stack(history['values'])
                    rewards_mem = torch.stack(history['reward'])
                    returns_mem = calculate_returns(rewards_mem, death_index, gamma=c_gamma)
                    advantages_mem = calculate_gae(rewards_mem, values_mem, death_index, 
                                                   gamma=c_gamma, gae_lambda=c_lambda)
                    first_run=False
                    preview=100
                    print('values: ', values_mem[:preview, 0, 0])
                    print('rewards: ', rewards_mem[:preview, 0])
                    print('returns: ', returns_mem[:preview, 0])
                    print('advantages: ', advantages_mem[:preview, 0])
                    print('actions: ', torch.stack(history['action'])[:preview, 0, 0])
                    all_returns.append(returns_mem[0, :].mean().item())
                else:
                    print('PLAY')
                    if first_run and os.path.exists(model_path):
                        model.load_state_dict(torch.load(model_path))
                    else:
                        if sim_count % student_update_step*2 == 0:
                            model.load_state_dict(model_student.state_dict())
                    if first_run and os.path.exists(critic_path):
                        model_critic.load_state_dict(torch.load(critic_path))
                    else:
                        if sim_count % student_critic_update_step*2 == 0:
                            model_critic_student.load_state_dict(model_critic.state_dict())
                    # model.reset_memories()
                    torch.save(model.state_dict(), model_path)
                    torch.save(model_critic.state_dict(), critic_path)
                    model.eval()
                    death_index = {i:[] for i in range(numEnvs)}
                    
                torch.manual_seed(sim_count//2)
                memories = {i:None for i in range(numEnvs)}
                cumulative_reward = torch.zeros(numEnvs, 1, device=device)
                returns_goal = initial_returns*torch.ones(numEnvs, device=device)
                birth_time = torch.zeros(numEnvs)
                for key in buffer.keys():
                    buffer[key]=[]
                for key in history.keys():
                    history[key]=history[key][-sim_duration:]
                backward_count, backward_critic_count = 0, 0
                sim_count+=1
                sim_step = 0
            sim_step+=1

            if sim_step == 2:
                env._task.post_reset()

            ### Observations: 'cartpole_view' 'obs_buf' (numEnvs, 4)

            all_obs = env._task.get_observations()['cartpole_view']
            obs = all_obs['obs_buf'].clone()
            update_buffer(buffer, 'obs', obs)
            history['obs'].append(obs)

            ### Video: 'cartpole_view' 'obs_buf' (numEnvs, 3, 256, 456)

            # vid = all_obs['video_buf'].clone()
            # update_buffer(buffer, 'video', vid)

            ### Reward (numEnvs, 1)

            env._task.calculate_metrics()
            reward=env._task.rew_buf.clone()
            update_buffer(buffer, 'reward', reward)
            history['reward'].append(reward)

            ### returns/ Return


            for agent_i in range(numEnvs):

                # Reset goal and memory when dying

                if env._task.reset_buf[agent_i].item() == 1:
                    returns_goal[agent_i] = initial_returns
                    memories[agent_i] = None

                    # Update death dictionnary during inference

                    if not replay_training:
                        death_index[agent_i].append(sim_step)

            ### Process input

            inputs={}
            inputs['video'] = torch.zeros(numEnvs, 3, nb_frames, 256, 456, device=device)
            # inputs['video'] = rearrange(buffer_padding(buffer['video'], [3, 256, 456]), 'b t c h w -> b c t h w')
            inputs['audio'] = torch.zeros(numEnvs, nb_frames*1920, device=device)
            if len(history['action']) == 0:
                history['action'].append(torch.zeros(numEnvs, cf.body_outputs['actions'], device=device))


            ### Agent
            
            new_actions = []
            inference_done = False
            if len(buffer['action'])==0 :
                inference_done = True
                past_horizon=torch.max(-nb_frames*torch.ones(numEnvs), 
                                torch.min(sim_step-birth_time, -1*torch.ones(numEnvs)))
                mask = repeat(torch.arange(nb_frames), '... -> b ...', b=numEnvs) < repeat(past_horizon, '... -> ... t', t=nb_frames)
                inputs['observations'] = buffer_padding(buffer['obs'], [cf.body_parts['observations']])
                inputs['observations'][mask] *= 0
                # batch = input_to_tensor(inputs, model.modality, cf.body_parts)
                dt=1

                with torch.no_grad():
                    values = model_critic_student(inputs['observations'][:,-dt:])

                if replay_training:
                    # batch_training = input_to_tensor(inputs, model.modality, cf.body_parts)
                    advantages = rearrange(advantages_mem[sim_step:sim_step+dt], 't b -> b t')

                    ## Critic

                    # returns = buffer_padding(buffer['returns_discount'], [1])[:,-dt:]
                    returns = rearrange(returns_mem[sim_step:sim_step+dt] , 't b -> b t')
                    values_training = model_critic(inputs['observations'][:,-dt:])
                    # returns = (returns - returns_mean) / (returns_std + eps)
                    critic_loss = F.mse_loss(values_training, returns)/accumulation_steps
                    critic_loss.backward()
                    backward_critic_count +=1
                    if backward_critic_count % accumulation_steps == 0 and backward_critic_count != 0:
                        backward_critic_count = 0
                        optimizer_critic.step()
                        optimizer_critic.zero_grad()
                    all_crit.append(critic_loss.item())


                ## Agent inference and memory update
                            
                output = {}
                output['actions'] = model(inputs['observations'][:,-dt:])
                actions = torch.tanh(output['actions'])
                mean, stddev = actions[:,:, 0], (1+actions[:,:,1])/4+c_min_std
                # shift=(actions[1:]-mean).mean(dim=0)
                dist = Normal(mean, stddev)
                sample, interpol= dist.sample().detach(), torch.linspace(0, 1, frames_future, device=device)
                output['actions'] = sample
                output['actions'] = output['actions'].detach()
                output['logprobs'] = dist.log_prob(output['actions'])
                new_actions=rearrange(output['actions'], 'b t -> b t 1')
                # memories[agent_i] = model.return_memories(device=device)

                ## Compute loss

                if replay_training:

                    ## Student

                    # !!!!!!!! Warning, delete 0 later !!!!!!
                    # loss = aux_loss * 0
                    output_training = {}
                    output_training['actions'] = model_student(inputs['observations'][:,-dt:])
                    actions = torch.tanh(output_training['actions'])
                    mean, stddev = actions[:,:, 0], (1+actions[:,:,1])/4+c_min_std
                    dist = Normal(mean, stddev)

                    # Statistics
                    logprobs = dist.log_prob(output['actions'])
                    entropy = dist.entropy()
                    # advantage = advantages[agent_i].detach()

                    # Loss
                    ratios = torch.exp(logprobs - output['logprobs'].detach())
                    surr1 = ratios * advantages
                    surr2 = torch.clamp(ratios, 1-eps_clip, 1+eps_clip) * advantages
                    loss = -torch.min(surr1, surr2) - c_entropy * entropy 
                    loss = loss.mean()/accumulation_steps
                    loss.backward()
                    backward_count += 1


                # Step optimizer

                if replay_training:
                    if backward_count % accumulation_steps == 0 and backward_count != 0:
                        optimizer.step()
                        optimizer.zero_grad()
                        backward_count = 0
                    all_losses.append(loss.item())

                # Update buffers

                new_actions = list(torch.unbind( rearrange(new_actions, 'b t d -> t b d') ))
                buffer['action'] += new_actions
                history['action'] += new_actions
                history['values'] += list(torch.unbind( rearrange(values, 'b t () -> t b ()') ))


            ## Print

            # Current action (numEnvs, 1)

            if(len(buffer['action'])>0):
                action = buffer['action'].pop(0)
                action = torch.clamp(action, min=-1, max=1)
            else:
                action = torch.dzeros(numEnvs, 1)

            # if( inference_done ):
            #     print('$$$$$$  INF  $$$$$$')
            #     print('obs ', inputs['observations'][agent_i,-1])
            #     print('actions: ', new_actions)
            #     # print('returns: ', buffer['returns_discount'][-1])
            #     print('values: ', values)
            #     if replay_training:
            #         print('advantages: ', advantages)

            # if env.sim_frame_count % sim_duration == 1:
            #     print("$$$$$$$$$$$$ 00000 $$$$$$$$$$$")
            #     if replay_training:
            #         all_returns.append(np.mean(returns_tmp))
            #         print('returns: ', returns_tmp)
            #         print('returns: ', returns)
            #         print('action: ', action)
            #         print('observations',inputs['observations'][-1][-1])
            #         print('loss', np.mean(all_losses[-100:]))
            

            # Step environment

            env._task.pre_physics_step(action.to(task.rl_device))
            env._world.step(render=render)
            env.sim_frame_count += 1
            env._task.post_physics_step()

        else:
            env._world.step(render=render)

    env._simulation_app.close()


if __name__ == '__main__':
    sim_duration = 264
    numEnvs = 8
    learning_rate = 0.0001
    learning_rate_critic = 0.0005
    initial_returns = 200
    accumulation_steps = 16
    c_entropy = 0
    student_critic_update_step = 2
    student_update_step = 4
    eps=1e-9
    c_gamma=0.99
    c_lambda=0.95
    c_min_std = 1e-5
    eps_clip=0.2
    model_path = 'weights/cartpole.pt'
    critic_path = 'weights/critic.pt'
    with initialize(config_path="memoryPerceiver/OmniIsaacGymEnvs/omniisaacgymenvs/cfg"):
        cfg = compose(config_name="config", overrides=["task=Cartpole", "headless=True","+numEnvs="+str(numEnvs)])
    parse_hydra_configs(cfg)

/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'config': Defaults list is missing `_self_`. See https://hydra.cc/docs/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
/isaac-sim/kit/python/lib/python3.7/site-packages/hydra/_internal/defaults_list.py:412: UserWarning: In config: Invalid overriding of hydra/job_logging:
Default list overrides requires 'override' keyword.
See https://hydra.cc/docs/next/upgrades/1.0_to_1.1/defaults_list_override for more information.

  deprecation_warning(msg)


Starting kit application with the following args:  ['/isaac-sim/exts/omni.isaac.kit/omni/isaac/kit/simulation_app.py', '/isaac-sim/apps/omni.isaac.sim.python.gym.headless.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=False', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--portable', '--no-window', '--allow-root']
Passing the following args to the base kit application:  ['-f', '/root/.local/share/jupyter/runtime/kernel-4cca15dd-96a7-44bd-9243-2dab33c39d3e.json']
[Warning] [omni.isaac.kit.simulation_app] Modules: ['omni.isaac.kit.app_framework'] were loaded before SimulationApp was sta

/isaac-sim/kit/exts/omni.graph/omni/graph/core/_impl/autonode/type_definitions.py:12: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  Float = NewType("Float", np.float)


],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2023-08-16 00:19:44 [1,891ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-08-16 00:19:44 [1,891ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
[1.911s] [ext: omni.kit.renderer.capture-0.0.0] startup
[1.915s] [ext: omni.kit.renderer.imgui-0.0.0] startup
2023-08-16 00:19:44 [1,903ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-08-16 00:19:44 [1,903ms] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.3]) (impl: carb.windowing-glfw.plugin)
2023-08-16 00:19:44 [1,904ms] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2023-08-16 00:19:44 [1,904ms] [Warning] [carb] Failed to startup plugin carb.windowing-glf

/isaac-sim/kit/kernel/py/omni/kit/app/_impl/__init__.py:104: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated since Python 3.3,and in 3.9 it will stop working
  if not isinstance(setting_py, str) and isinstance(setting_py, collections.Sequence):


...

dof pos tensor([[-0.3858,  0.0437]], device='cuda:0')
dof pos tensor([[ 0.5076, -0.0733]], device='cuda:0')
dof pos tensor([[-0.6629, -0.3187]], device='cuda:0')
dof pos tensor([[0.8027, 0.0228]], device='cuda:0')
dof pos tensor([[ 0.3012, -0.1934]], device='cuda:0')
dof pos tensor([[0.1349, 0.3182]], device='cuda:0')
dof pos tensor([[0.0075, 0.1841]], device='cuda:0')
dof pos tensor([[ 0.4986, -0.2166]], device='cuda:0')
dof pos tensor([[-0.5807,  0.1578]], device='cuda:0')
dof pos tensor([[0.8816, 0.0532]], device='cuda:0')
dof pos tensor([[0.2527, 0.0943]], device='cuda:0')
dof pos tensor([[ 0.5200, -0.3012]], device='cuda:0')
dof pos tensor([[0.5007, 0.3265]], device='cuda:0')
dof pos tensor([[ 0.9208, -0.2617]], device='cuda:0')
dof pos tensor([[-0.5665,  0.2508]], device='cuda:0')
dof pos tensor([[-0.8854,  0.3015]], device='cuda:0')
dof pos tensor([[-0.1335,  0.0999],
        [-0.8784,  0.1936]], device='cuda:0')
dof pos tensor([[0.0566, 0.1950]], device='cuda:0')
dof pos tensor

/isaac-sim/kit/python/lib/python3.7/site-packages/ipykernel_launcher.py:248: UserWarning: Using a target size (torch.Size([8, 1])) that is different to the input size (torch.Size([8, 1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.


dof pos tensor([[-0.3947,  0.3036]], device='cuda:0')
dof pos tensor([[-0.5267,  0.2129]], device='cuda:0')
dof pos tensor([[ 0.3952,  0.2973],
        [-0.3974,  0.0861]], device='cuda:0')
dof pos tensor([[-0.6376, -0.0055]], device='cuda:0')
dof pos tensor([[ 0.8527, -0.2724]], device='cuda:0')
dof pos tensor([[-0.3696,  0.3162]], device='cuda:0')
dof pos tensor([[ 0.6603, -0.0558]], device='cuda:0')
dof pos tensor([[ 0.7761, -0.0024]], device='cuda:0')
dof pos tensor([[-0.7878, -0.3797]], device='cuda:0')
dof pos tensor([[-0.4835, -0.2437]], device='cuda:0')
dof pos tensor([[0.9207, 0.1172]], device='cuda:0')
dof pos tensor([[0.4399, 0.1600]], device='cuda:0')
dof pos tensor([[-0.8917,  0.2568]], device='cuda:0')
dof pos tensor([[-0.1741,  0.1323],
        [ 0.1124, -0.2716]], device='cuda:0')
dof pos tensor([[-0.2414, -0.3151],
        [ 0.3241, -0.0073]], device='cuda:0')
dof pos tensor([[-0.7595,  0.2898]], device='cuda:0')
dof pos tensor([[-0.9394,  0.0566]], device='cuda:0')
do

/isaac-sim/kit/python/lib/python3.7/site-packages/ipykernel_launcher.py:248: UserWarning: Using a target size (torch.Size([8, 1])) that is different to the input size (torch.Size([8, 1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.


dof pos tensor([[ 0.6792, -0.3609]], device='cuda:0')
dof pos tensor([[-0.1878,  0.0671],
        [-0.1712, -0.2742]], device='cuda:0')
dof pos tensor([[ 0.6293, -0.3135]], device='cuda:0')
dof pos tensor([[-0.4406, -0.2629]], device='cuda:0')
dof pos tensor([[0.8649, 0.1313]], device='cuda:0')
dof pos tensor([[-0.4225,  0.2988]], device='cuda:0')
dof pos tensor([[-0.9339,  0.1394]], device='cuda:0')
dof pos tensor([[0.7180, 0.1055]], device='cuda:0')
dof pos tensor([[ 0.4783, -0.3653]], device='cuda:0')
dof pos tensor([[-0.5744,  0.1467]], device='cuda:0')
dof pos tensor([[ 0.6701, -0.0023]], device='cuda:0')
dof pos tensor([[0.2427, 0.1883]], device='cuda:0')
dof pos tensor([[ 0.7599, -0.2141]], device='cuda:0')
dof pos tensor([[ 0.2690, -0.2451]], device='cuda:0')
dof pos tensor([[-0.8468,  0.2204],
        [ 0.4267, -0.2153]], device='cuda:0')
dof pos tensor([[ 0.6821, -0.0687]], device='cuda:0')
dof pos tensor([[0.0510, 0.2916]], device='cuda:0')
dof pos tensor([[-0.0164, -0.3587]

: 